# 분해기 게이트 완화 + 중복 task 병합 실험

케이스: `건강하고 건강하며 건강한데 또 건강했다가 건강하려다가 건강해야 해` → 현재 `out_of_scope`, 기대 `건강하기`.

현행에선 `task_splitter_node` 의 `_is_low_information` 압축 게이트(ratio 0.670 < 0.75)가 **LLM 호출 전에** 잘라 `out_of_scope` 로 보낸다. 이 게이트는 `ㅋㅋㅋ` 류 순수 반복 쓰레기를 거르려고 의도적으로 넣은 것이라, '같은 실단어를 강조 반복한 입력'과 '노이즈'를 압축률만으로는 구분하지 못한다.

이 노트북은 두 가지를 본다:
1. **게이트 완화** — `ports.llm.split_tasks` 를 직접 부르면 노드의 게이트를 우회한다(게이트는 노드에 있고 어댑터엔 없다). 즉 '게이트가 없었다면 LLM 이 뭘 내놓는가'를 그대로 관찰한다.
2. **중복 병합** — LLM 이 `건강하기` 를 여러 번 뱉으면(`[건강하기, 건강하기, ...]`) 제목 기준으로 하나로 합칠 수 있는가.

control 로 진짜 garbage(`ㅋㅋㅋ`, `아아아`)와 정상 다중 task 를 함께 돌려, 게이트를 풀었을 때/dedup 했을 때 부작용을 확인한다.

In [ ]:
import os
import sys
import pathlib

_root = pathlib.Path.cwd()
while not (_root / "agents").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

_env = _root / ".env"
if _env.is_file():
    for line in _env.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

print("repo root =", _root, "| LLM_PROVIDER =", os.environ.get("LLM_PROVIDER"))

In [ ]:
from datetime import date

from api.config import AppConfig
from api.deps import build_todo_generate_ports

cfg = AppConfig.from_env()
ports = build_todo_generate_ports(cfg)
TODAY = date.today()
print("ports.llm:", type(ports.llm).__name__, "| TODAY:", TODAY)

In [ ]:
from agents.todo_creation.todo.nodes.task_splitter import _is_low_information


def _norm(title: str) -> str:
    return "".join(title.split()).lower()


def dedup_tasks(tasks):
    """제목이 같은 task 를 첫 등장 순서로 하나만 남긴다.
    ponytail: 정규화 제목 완전일치 기준. 의미 유사(헬스/운동)는 안 합친다 — 필요하면 임베딩으로."""
    seen, out = set(), []
    for t in tasks:
        k = _norm(t.title)
        if k in seen:
            continue
        seen.add(k)
        out.append(t)
    return out


CASES = [
    "건강하고 건강하며 건강한데 또 건강했다가 건강하려다가 건강해야 해",  # 핵심: 같은 실단어 강조 반복
    "ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ",                                       # 진짜 garbage — out_of_scope 유지돼야
    "아아아아아아아아아아아아",                                    # 진짜 garbage
    "장보고 운동해야지 내일은 친구들 만나기로 했어",                 # 정상 다중 task — dedup 이 잘못 합치면 안 됨
    "두쫀쿠 사고 병원가야지",                                      # case1 참고(사고/사기는 별개 이슈)
]

In [ ]:
for prompt in CASES:
    print("=" * 72)
    print("입력:", prompt)
    print(f"현행 게이트 _is_low_information = {_is_low_information(prompt)}  (True 면 LLM 호출 없이 out_of_scope)")
    try:
        sr = await ports.llm.split_tasks(prompt=prompt, today=TODAY)
    except Exception as e:
        print(f"  ! split_tasks 실패: {type(e).__name__}: {e}")
        continue
    print("게이트 우회 시 LLM intent =", sr.intent)
    raw = [t.title for t in sr.tasks]
    merged = [t.title for t in dedup_tasks(sr.tasks)]
    print(f"  raw   ({len(raw)}): {raw}")
    print(f"  dedup ({len(merged)}): {merged}")

## 읽는 법

- **건강 반복** 행: 게이트 = `True`(그래서 현행은 out_of_scope). 게이트를 풀었을 때 LLM intent 가 `plan` + `건강하기` 면 → '게이트 완화 + dedup' 으로 살릴 수 있다는 뜻. 만약 LLM 이 그래도 `out_of_scope` 거나 garbage 면 → 게이트만 풀어선 안 되고 LLM/SFT 손이 필요하다는 뜻.
- **ㅋㅋㅋ / 아아아** 행: 게이트를 풀면 이게 어떻게 처리되는지가 게이트 완화의 비용이다. LLM 이 알아서 `out_of_scope` 면 게이트를 LLM 으로 대체 가능, garbage task 를 만들면 게이트가 여전히 필요하다는 근거.
- **장보기** 행: dedup 컬럼이 서로 다른 task(장보기/운동/친구 만나기)를 잘못 합치지 않는지 확인(정규화 제목 완전일치라 안전해야 정상).

결론에 따라: (a) 게이트 제거+dedup 만으로 충분 → 코드 소폭 수정, (b) LLM 이 안 따라줌 → 프롬프트 예시 추가 또는 SFT.